In [1]:
# ──────────────────────────────────────────────────────────────
# Headers are in the code because they look nice
# "If my commit messages don't have emojis, how would you know how I feel?"
# - ProgrammersAreAlsoHuman, '0.1x engineers'
# ──────────────────────────────────────────────────────────────

"""This code trains several models to perform a 3SUM task generated
by https://github.com/JacobPfau/fillerTokens/tree/master, comparing
model training curves, attention maps, and test set performance in
order to highlight the role of filler tokens."""

# I don't think we need these:
# !pip uninstall -y torch torchvision torchaudio
# !pip install -U bitsandbytes accelerate transformers peft

# Try/except to avoid repeated install; not sure of a better way
try:
    import bitsandbytes as bnb
except:
    !pip install --prefer-binary bitsandbytes

    !pip uninstall -y datasets fsspec gcsfs
    !pip install -U "datasets>=2.19.1" "fsspec>=2024.3.1"


In [21]:
# ──────────────────────────────────────────────────────────────
# Imports
# ──────────────────────────────────────────────────────────────
import datetime as dt
import itertools
import json
import os
import tempfile
import pathlib
import shutil
import re
import textwrap
import subprocess
import shlex
from pathlib import Path
from glob import glob
from functools import partial
from typing import Final
from types import SimpleNamespace as ns
from collections import Counter

import numpy as np
import pandas as pd
import polars as pl
import torch
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset, load_from_disk
from google.colab import drive
from huggingface_hub import notebook_login
from peft import LoraConfig, get_peft_model, PeftModel
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)


In [3]:
# ──────────────────────────────────────────────────────────────
# Global variables and namespace for the run
# ──────────────────────────────────────────────────────────────
# Dataset settings
MATCH_LENGTH = 12
DIMENSION = 3

# Run namespace
presets = {
    'direct': ns(
        name='direct',
        filler_tok=None,
        generate_raw=True,
        cot_rate=0,
        no_filler_rate=1
    ),
    'cot': ns(
        name='cot',
        filler_tok=None,
        generate_raw=True,
        cot_rate=1,
        no_filler_rate=1
    ),
    'dot': ns(
        name='dot',
        filler_tok=' .',
        generate_raw=True,
        cot_rate=0,
        no_filler_rate=0
    ),
    'low_S': ns(
        name='low_S',
        filler_tok='big',
        generate_raw=False
    ),
    'high_S': ns(
        name='high_S',
        filler_tok='neu',
        generate_raw=False
    )
}
RUN = presets['direct']

# Output paths
BASE_DIR = "/content/drive/MyDrive/Colab_Files/repurposed_tokens"
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"  # or 3-8B
MODEL_NAME = MODEL_ID.split('/')[-1]
RUN.dir = os.path.join(BASE_DIR, MODEL_NAME, RUN.name)
MODEL_DIR = os.path.join(RUN.dir, 'model')
os.makedirs(MODEL_DIR, exist_ok=True)


In [14]:
# ──────────────────────────────────────────────────────────────
# Github and Drive setup
# ──────────────────────────────────────────────────────────────
%cd /content

# Mount drive for data
drive.flush_and_unmount()
if os.path.exists('/content/drive') and not os.path.islink('/content/drive'):
    shutil.rmtree('/content/drive')
drive.mount('/content/drive', force_remount=True)

# Filepaths
FILLER_DIR = "/content/fillerTokens"
ENTROPIX_DIR = "/content/entropix"

# Clone and setup repos
if not os.path.exists(FILLER_DIR):
    !git clone https://github.com/BreckEmert/fillerTokens.git
    !pip install -r {FILLER_DIR}/requirements.txt

if not os.path.exists(ENTROPIX_DIR):
    !git clone https://github.com/BreckEmert/entropix.git

# Create symlink to data
if os.path.islink('./data') or os.path.exists('./data'):
    os.unlink('./data')
os.symlink(RUN.dir, './data')

# Set HF cache
os.environ['HF_HOME'] = '/content/drive/MyDrive/HF_cache'


/content
Mounted at /content/drive


In [5]:
# ──────────────────────────────────────────────────────────────
# HF setup
# ──────────────────────────────────────────────────────────────
notebook_login()


# 3SUM Dataset Generation Script

In [23]:
"""
I plan to use these training runs:
    1) CoT as filler (filler unmasked)
    2) Dots as filler (filler masked out)
    3) Low entropy tokens as filler (filler masked out)
    4) High entropy tokens as filler (filler masked out)
    5) Direct-to-answer

This allows for
    1) A high-quality gold standard for the model, direct-to-answer.  Not very related to my experiment but a super-baseline.
    2) Progressively more difficult tasks to pull out some sort of linear relationship (heavily relies on the assumption that I've picked tokens which are linear along some x axis, but these *were* calculated via the entropix repo as progressively more-difficult-to-predict tokens.  Need to formalize this more but for now not worth the time).  Keep the experiment simple as there are so many possible experiments.


Outline of data pipeline for clarity:
1) We generate three kinds of samples, direct, CoT and punct:
    533 569 530 814 A False     VERIFY THIS DOESNT HAVE P AND A IM NOT SURE!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    371 578 006 519 P 1- 4 0- 7 3- 7- 4- 4 5- 6 A True
    873 545 827 245 P . . . . . . . . . . . . . A False
    # Note that this is prompt followed by answer in the same string
    # Answers are one word long (True/False), so
      future [-1], len(prompt)-1, are referring to this
2) Turn that into a HF dataset which has two columns:
    a) input_ids: tokenized full prompts
    b) labels: same except -100-masked everywhere but answer and eos tokens
3) Lastly, the collator recieves a list of those dicts and
    a) pads batches to the longest of that batch
    b) stacks examples into tensors
    c) leaves labels alone
"""

# ──────────────────────────────────────────────────────────────
# Generate raw math data with scripts.data_match3
# ──────────────────────────────────────────────────────────────
# This must be run because the script is a module:
%cd {FILLER_DIR}

# Get output paths and see if they exist
raw_train_csv = os.path.join(BASE_DIR, f"train_{RUN.name}.csv")
raw_test_csv = os.path.join(BASE_DIR, f"test_{RUN.name}.csv")
# cfg_path = os.path.join(BASE_DIR, f"config_{RUN.name}.csv")  # uncomment these in another run its too late for this run I'm not rerunning the scripts haha

if bool(glob(raw_train_csv)) and bool(glob(raw_test_csv)):
    print(f"Datasets already exists at: \n{raw_train_csv}\n{raw_test_csv}")
else:
    raise ValueError()  # While you're figuring out filepathing this avoids accidental script runs

    # Run the data generation script
    cmd = (
        f"python -m scripts.data_match3 "
        f"--name demo3s_{name} "
        f"--length {MATCH_LENGTH} "
        f"--dimension {DIMENSION} "
        f"--train_samples 500000 "
        f"--test_samples 10000 "
        f"--cot_rate {settings.cot_rate} "
        f"--no_filler_rate {settings.no_filler_rate} "
        f"--corruption_rate 1.33 "
        f"--data_path {BASE_DIR}"
    )
    print("Generating ", name, cmd)
    subprocess.run(shlex.split(cmd))
    # Args:
        # --length 12  # number of tuples per instance
        # --dimension 3  # number of integers per tuple
        # --cot_rate 0.5  # fraction to annotate with CoT traces
        # --no_filler_rate 0  # 0 so every non-CoT is punctuation-filler
        #  --corruption_rate 1.33  # False instances have avg 1.33 digits randomly replaced (False are generated from True examples and made False by replacing digits)
    # Generates:
        # args_demo3s_YYYY-MM-DD.json  # metadata
        # demo3s_trainset_YYYY-MM-DD.csv
        # demo3s_testset_YYYY-MM-DD.csv

    # Remove the date from their names
    raw_train_csv = glob(f"{BASE_DIR}/demo3s_trainset_*.csv")[0]
    raw_test_csv = glob(f"{BASE_DIR}/demo3s_testset_*.csv")[0]
    # raw_cfg_json = glob(f"{BASE_DIR}/args_demo3s_*.json")[0]

print(f"Train csv: {pl.scan_csv(raw_train_csv).describe()}\n")
print(f"Test csv: {pl.scan_csv(raw_test_csv).describe()}\n")
# print(f"Config json: {open(cfg_json).read()}")

!ls {BASE_DIR}


/content/fillerTokens
Datasets already exists at: 
/content/drive/MyDrive/Colab_Files/repurposed_tokens/train_direct.csv
/content/drive/MyDrive/Colab_Files/repurposed_tokens/test_direct.csv
Train csv: shape: (9, 2)
┌────────────┬─────────────────────────────────┐
│ statistic  ┆  112 180 381 567 280 143 698 4… │
│ ---        ┆ ---                             │
│ str        ┆ str                             │
╞════════════╪═════════════════════════════════╡
│ count      ┆ 499999                          │
│ null_count ┆ 0                               │
│ mean       ┆ null                            │
│ std        ┆ null                            │
│ min        ┆  000 000 237 736 967 372 492 4… │
│ 25%        ┆ null                            │
│ 50%        ┆ null                            │
│ 75%        ┆ null                            │
│ max        ┆  999 997 981 112 746 246 651 9… │
└────────────┴─────────────────────────────────┘

Test csv: shape: (9, 2)
┌────────────┬───────────

In [24]:
# ──────────────────────────────────────────────────────────────
# Generate low-S and high-S datasets as well
# ──────────────────────────────────────────────────────────────
if RUN.name in ["low_S", "high_S"]:
    def replace_filler(prompt: str, filler_token: str) -> str:
        # Get segments of the example
        question, right = prompt.split(" P", 1)
        filler, answer = prompt.split(" A", 1)

        # Make the new filler
        n_tokens = len(filler.split())
        new_filler = " ".join(filler_token for _ in range(n_tokens))

        # Return the original q/a with the new filler
        return f"{question} P {new_filler} A {answer}"  # AGAIN, VERIFY SPACES

    # Paths to the datasets with replaced-filler in RUN.dir
    train_path = os.path.join(RUN.dir, f"{RUN.name}_trainset.csv")
    test_path = os.path.join(RUN.dir, f"{RUN.name}_testset.csv")

    for raw_path, out_path in [
        (raw_train_csv, train_path),
        (raw_test_csv, test_path)
    ]:
        # Don't regenerate it if it exists
        if os.path.exists(out_path):
            continue

        # Run replace_filler
        df = (
            pl.scan_csv(raw_path)
            .with_columns(
                pl.col('prompt')
                .map_elements(lambda s: replace_filler(s, RUN.filler_tok))
            )
        )

        # Write to RUN.dir
        df.sink_csv(out_path)
        print(f"{RUN.name} dataset generated at {out_path}")
        print(df.head())
else:
    print("No generation done, not for this RUN")


No generation done, not for this RUN


# Tokenization and HF Dataset Creation

In [25]:
# ──────────────────────────────────────────────────────────────
# Prepare tokenizer
# ──────────────────────────────────────────────────────────────
# Note: We only want loss on the answer token and eos, because we want it to use the periods how it wants, not generate them.

# Instantiate the llama tokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tok.pad_token = tok.eos_token

# Now we can set the ids of important constants
if RUN.filler_tok:
    filler_ids = tok(RUN.filler_tok, add_special_tokens=False)['input_ids']
    RUN.filler_id = torch.tensor(filler_ids).unique()

COT_BEGIN_ID = tok(" P", add_special_tokens=False)['input_ids']
COT_END_ID = tok(" A", add_special_tokens=False)['input_ids']
TRUE_ID = tok(" True", add_special_tokens=False)['input_ids']
FALSE_ID = tok(" False", add_special_tokens=False)['input_ids']
assert len(COT_BEGIN_ID + COT_END_ID + TRUE_ID + FALSE_ID) == 4

print("COT_BEGIN_ID: ", COT_BEGIN_ID)
print("COT_END_ID: ", COT_END_ID)
print("TRUE_ID: ", TRUE_ID)
print("FALSE_ID: ", FALSE_ID)


# ──────────────────────────────────────────────────────────────
# Prepare raw math data into a HF dataset
# ──────────────────────────────────────────────────────────────
# Paths to the tokenized dir
train_tok_dir = os.path.join(RUN.dir, f"{RUN.name}_trainset_tokenized")
test_tok_dir = os.path.join(RUN.dir, f"{RUN.name}_testset_tokenized")

# Only tokenize the raw csvs if they haven't been made
if os.path.exists(train_tok_dir) and os.path.exists(test_tok_dir):
    print(f"Loading existing ds at:\n{train_tok_dir}\n{test_tok_dir}")
    train_ds = load_from_disk(train_tok_dir)
    test_ds = load_from_disk(test_tok_dir)
else:
    # Grab the data
    datasets = load_dataset(
        'csv',
        data_files={'train': train_path, 'test': test_path},
        header=None,
        column_names=['prompt']
    )
    train_raw = datasets['train']
    test_raw = datasets['test']

    def tokenize_and_mask(batch):
        # Tokenize the batch
        ids_batch = tok(batch['prompt'], add_special_tokens=True)['input_ids']
            # Note this only adds bos, NOT eos.

        # Mask off everything but answer
        labels = []
        for ids in ids_batch:
            label = [-100]*len(ids)
            # If we wanted the model to produce CoT, we could do [MATCH_LENGTH:] mask instead, but I don't think that helps the research question.
            label[-1] = ids[-1]  # We asserted the answer is 1 id only
            labels.append(label)

        return {'input_ids': ids_batch, 'labels': labels}

    train_ds = train_raw.map(tokenize_and_mask, batched=True,
                             remove_columns=['prompt'])
    test_ds = test_raw.map(tokenize_and_mask, batched=True,
                           remove_columns=['prompt'])

    # Save to Drive for next time
    train_ds.save_to_disk(train_tok_dir)
    test_ds.save_to_disk(test_tok_dir)

# Collation is done in the finetune step, which just pads

# Peek at the first rows and make sure we have the right set
for split, ds in [("Train", train_ds), ("Test", test_ds)]:
    print(f"Sample of {split} ds:")
    for ex in ds.select(range(2)):
        ids = ex["input_ids"]
        print(ids)
        print(tok.decode(ids, skip_special_tokens=False), "\n")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


COT_BEGIN_ID:  [393]
COT_END_ID:  [362]
TRUE_ID:  [3082]
FALSE_ID:  [3641]


FileNotFoundError: Unable to find '/content/drive/MyDrive/Colab_Files/repurposed_tokens/Llama-3.2-1B-Instruct/direct/train_direct.csv'

# Config/Collator for both Finetune AND Eval Runs

In [8]:
# ──────────────────────────────────────────────────────────────
# Config/Collator for both finetune AND eval runs
# ──────────────────────────────────────────────────────────────
# Bnb config
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # Run this on T4s
)

# Just pad/stacks
    # DCWP defaults to -100 for padding, so we're good there.
    # HF documentation is unacceptable and trying to get their stuff to work is proving to be the worst use of time ever so I'm removing it
# data_collator = DataCollatorWithPadding(
#     tokenizer=tok,
#     pad_to_multiple_of=8,  # Only useful on A100
#     return_tensors='pt',  # I'm from tf so I'll be typing this :D
# )
def pad_batch(sequences, pad_id):
    return pad_sequence(
        [torch.tensor(seq, dtype=torch.long) for seq in sequences],
        batch_first=True, padding_value=pad_id
    )

def custom_data_collator(examples):
    input_ids = pad_batch([ex['input_ids'] for ex in examples],
                          tok.pad_token_id)
    labels = pad_batch([ex['labels'] for ex in examples], -100)
    return {
        'input_ids': input_ids,
        'labels': labels,
        'attention_mask': (input_ids != tok.pad_token_id).long()
    }


# Finetune

In [9]:
"""
I plan to use these training runs:
    1) CoT as filler (filler unmasked)
    2) Dots as filler (filler masked out)
    3) Low entropy tokens as filler (filler masked out)
    4) High entropy tokens as filler (filler masked out)
    5) Direct-to-answer

This allows for
    1) A high-quality gold standard for the model.  Not very
        related to my experiment but a super-baseline.
    2) Progressively more difficult tasks to highlight the
        effect of filler tokens without the large amount of
        runs that are theoretically all relevant here, as
        we step through runs 2-4.
"""
# ──────────────────────────────────────────────────────────────
# Load the base model
# ──────────────────────────────────────────────────────────────
# Load model and resize embeddings
# !pip install -U flash-attn --no-build-isolation
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    torch_dtype='auto',
    device_map='auto',
    attn_implementation='sdpa',  # Eager for eval
)
base_model.resize_token_embeddings(len(tok))

# NO LONGER DOING THIS:
# Enabling checkpointing test
# base_model.gradient_checkpointing_enable()
# base_model.config.use_cache = False
base_model.config.use_cache = True


# ──────────────────────────────────────────────────────────────
# Attach LoRA adapter to the model
# ──────────────────────────────────────────────────────────────
# LoRA config
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
)
# You may wonder if targeting attn is self-fulfilling for my experiment.  It is a decent bias, but, because attention mass != usefulness, indirect paths exist, and the rest of the model must adapt to any changes in attn, it's not locking in the results.  I should test LoRA on MLP and compare, ideally, which maybe I'll get around to but I only have so much compute and time.
# MLP would be gate_proj, up_proj, down_proj
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

# Now that the model is made we can move this to the gpu
if RUN.filler_tok:
    RUN.filler_id.to(model.device)


trainable params: 1,703,936 || all params: 1,237,518,336 || trainable%: 0.1377


In [10]:
# ──────────────────────────────────────────────────────────────
# Finetune
# ──────────────────────────────────────────────────────────────
args = TrainingArguments(
    output_dir=os.path.join(MODEL_DIR, 'checkpoints'),
    label_names=['labels'],
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,  # 8*4 = 32 examples per opt step
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    max_steps=3_000,
    warmup_steps=200,
    bf16=False,
    fp16=True,
    fp16_full_eval=True,
    optim='paged_adamw_8bit',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='no',
    report_to='none',  # DISABLE WANDB FOR QUICK RUNS
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=custom_data_collator,
)

# Train
trainer.train()
model.save_pretrained(MODEL_DIR)
tok.save_pretrained(MODEL_DIR)


Epoch,Training Loss,Validation Loss
0,0.323000,0.330147


('/content/drive/MyDrive/Colab_Files/repurposed_tokens/Llama-3.2-1B-Instruct/direct/model/tokenizer_config.json',
 '/content/drive/MyDrive/Colab_Files/repurposed_tokens/Llama-3.2-1B-Instruct/direct/model/special_tokens_map.json',
 '/content/drive/MyDrive/Colab_Files/repurposed_tokens/Llama-3.2-1B-Instruct/direct/model/chat_template.jinja',
 '/content/drive/MyDrive/Colab_Files/repurposed_tokens/Llama-3.2-1B-Instruct/direct/model/tokenizer.json')

# Evaluation and Visualization

In [30]:
# ──────────────────────────────────────────────────────────────
# Load the finetuned model and fuse in the LoRA adapter
# ──────────────────────────────────────────────────────────────
# Base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    torch_dtype='auto',
    device_map='auto',
    attn_implementation='eager',  # for pure accuracy eval we can do sdpa on T4 or flash_attention_2 on A100
)

# Finetuned LoRA
model = PeftModel.from_pretrained(
    base_model,
    MODEL_DIR,
    is_trainable=False,
    torch_dtype='auto',
    device_map='auto',
)

# Merge
model = model.merge_and_unload()
model.eval()

# Compile
model = torch.compile(model, mode="reduce-overhead")
model.config.pad_token_id = tok.pad_token_id


In [12]:
# ──────────────────────────────────────────────────────────────
# Attention hook
# ──────────────────────────────────────────────────────────────
def _record_attn(layer_idx, module, input, output):
    """Writes (B, H) into CURRENT_FILL (B, L, H)"""
    attention_probs = output[1]  # (B, H, Q, K)

    filler_mask = torch.isin(
        TOKEN_IDS_THIS_BATCH.to(model.device),
        RUN.filler_id.to(model.device)
    )[:, None, None, :]  # (B, 1, 1, K)
        # B matches
        # 1 broadcasts across all heads
        # 1 broadcasts across all query positions
        # K matches

    # Compute mean over query dimension
    fill = (attention_probs * filler_mask).sum(-1).mean(-1)  # (B, H)
    CURRENT_FILL[:, layer_idx, :] = fill.cpu()

# Attach to every layer
def attach_attention_hooks(model):
    return [
        block.self_attn.register_forward_hook(partial(_record_attn, layer_idx)) for layer_idx, block in enumerate(model.model.layers)
    ]

L = model.config.num_hidden_layers
H = model.config.num_attention_heads
handles = attach_attention_hooks(model)


In [ ]:
# ──────────────────────────────────────────────────────────────
# Attention logging and visualization
# ──────────────────────────────────────────────────────────────
def get_raw_attention(dataloader):
    global TOKEN_IDS_THIS_BATCH, CURRENT_FILL
    ATTN_BUCKET.clear()

    with torch.no_grad():
        for batch in tqdm(dataloader):
            TOKEN_IDS_THIS_BATCH = batch["input_ids"]  # (B, S)

            B = TOKEN_IDS_THIS_BATCH.size(0)  # L and H are global
            CURRENT_FILL = torch.zeros(B, L, H)  # (B, L, H)
                # Remember CURRENT_FILL is filled by the hooks we set
            _ = model(
                **{k: v.to("cuda") for k, v in batch.items()},
                use_cache=False,
                output_attentions=True
            )
            ATTN_BUCKET.append(CURRENT_FILL)

    raw_attn = torch.cat(ATTN_BUCKET, dim=0)  # (N, L, H)
    return raw_attn

def calculate_enrichment(raw_attn):
    """Adjusts the attention for the percentage of filler tokens
    in the ds otherwise the attention mass is biased
    """
    # Calculate the filler rate
    n_fill, n_total = 0, 0
    for ex in test_ds:
        ids = ex["input_ids"]
        n_total += len(ids)
        n_fill += torch.isin(torch.tensor(ids), RUN.filler_id).sum().item()

    fill_rate = n_fill / n_total if n_total else 0.0
    print(f"Corpus filler rate: {fill_rate:.3%}")
        # THIS MIGHT BE WRONG FOR DIRECT?  BCAUSE THERES NO FILLER TO DILUTE  not sure what I'm looking for with direct

    # Calculate the enrichment
    prob_to_fill = raw_attn.mean(dim=0)  # (L, H)
    return prob_to_fill / max(fill_rate, 1e-9)

def plot_attention(mat, title=''):
    """Quick heat-map"""
    import matplotlib.pyplot as plt
    plt.imshow(mat.numpy(), aspect='auto')
    plt.xlabel("Head")
    plt.ylabel("Layer (0 = bottom)")
    plt.title(title)
    plt.colorbar(label="× corpus filler rate")
    plt.show()

# Get raw attention
ATTN_BUCKET = []
raw_attn = get_raw_attention(
    torch.utils.data.DataLoader(
        test_ds.shuffle().select(range(500)),
        batch_size=8,
        shuffle=False,
        collate_fn=custom_data_collator,
    )
)

# Convert to enrichment
    # I regret writing mean_attn because I don't even think I want these plots yet for direct runs.  Might be used later, I guess.  But for run = 'direct' I'm just moving to accuracy.
if RUN.filler_id is not None:
    enr = calculate_enrichment(raw_attn)
    plot_attention(enr, f"Enrichment - {RUN.name}")
    np.save(os.path.join(RUN.dir, "enrichment.npy"), enr.numpy())
else:
    mean_attn = raw_attn.mean(dim=0)
    plot_attention(mean_attn, f"Mean Attention - {RUN.name}")
    np.save(os.path.join(RUN.dir, "mean_attn.npy"), mean_attn.numpy())


In [33]:
# ──────────────────────────────────────────────────────────────
# Accuracy Eval
# ──────────────────────────────────────────────────────────────
# Remove these if they're still on
for handle in handles:
    handle.remove()

def accuracy(model, testset) -> float:
    """Evaluates model accuracy on test set"""
    # Iterate over test set
    invalid, correct = Counter(), 0
    with torch.no_grad():
        for ex in tqdm(testset, leave=False):
            # Forward the input to get an output
            ids = torch.tensor(ex['input_ids']).unsqueeze(0).to('cuda')
            mask = torch.ones_like(ids)
            print(ids)
            print(tok.decode(ids[0], skip_special_tokens=False))

            break
            # keep only the prompt up to (and incl.) the " P" delimiter, drop the filler stretch
            cut = (ids[0] == COT_BEGIN_ID[0]).nonzero(as_tuple=True)[0] + 1
            ids, mask = ids[:, :cut], mask[:, :cut]
            print(ids)
            # break

            out = model.generate(
                input_ids=ids[:, :-1],  # Don't give it the answers
                attention_mask=mask[:, :-1],
                max_new_tokens=1,
                do_sample=False,
                temperature=None,
                top_p=None,  # Trying these to None because warnings?
            )
            print(out)
            break

            # Check answer
            pred_id = out[0, -1].item()
            exp_id = ids[0, -1].item()

            if pred_id == exp_id:
                correct += 1
            elif pred_id not in [TRUE_ID[0], FALSE_ID[0]]:
                invalid[pred_id] += 1

    accuracy = correct / len(testset)
    return accuracy, invalid

# this is off now that I don't have dots/no dots since that would be handled by aggregating several runs of data, not something we do all at once necessarily?  I guess we can do it all at once after all the models are trained, but it seems useful to have some sort of utility function that works as we go so we can tune the runs.
acc_dots, invalid_dots = accuracy(model,                      .shuffle().select(range(100)))

print(f"Invalid count: {sum(invalid_dots)}")
print(f"Accuracy: {acc_dots:.3f}")


  0%|          | 0/100 [00:00<?, ?it/s]

tensor([[128000,    220,  19104,    220,  20899,    220,  16718,    220,  23388,
            220,  21985,    220,   8258,    220,  17014,    220,  26661,    220,
          11242,    220,  17014,    220,  22539,    220,  20785,    393,    220,
             15,     12,    220,     20,    220,     17,     12,    220,     15,
            220,     18,     12,    220,     22,    220,     15,     12,    220,
             23,    220,     20,     12,    220,     15,    220,     21,     12,
            220,     24,    220,     22,     12,    220,     17,    220,     15,
             12,    220,     22,    220,     15,     12,    220,     21,    220,
             15,     12,    220,     19,    220,     15,     12,    220,     15,
            220,     17,     12,    220,     16,    220,     18,     12,    220,
             17,    220,     19,     12,    220,     16,    220,     16,     12,
            220,     18,    220,     16,     12,    220,     24,    220,     16,
             12,    220,    

# Debug Cells

In [28]:
# Temp debugging cell
# AI GENERATED CELL
mc = model.config  # shorthand

# Ensure pad_token_id is set
if mc.pad_token_id is None:
    mc.pad_token_id = tok.pad_token_id

# IDs of interest
relevant_ids = {
    "TRUE_ID":       TRUE_ID[0],
    "FALSE_ID":      FALSE_ID[0],
    "COT_BEGIN_ID":  COT_BEGIN_ID[0],
    "COT_END_ID":    COT_END_ID[0],
    "pad_token_id":  mc.pad_token_id,
    # "eos_token_id":  mc.eos_token_id,
    "bos_token_id":  mc.bos_token_id,
    # "unk_token_id":  mc.unk_token_id,
    "space_id (220)": 220,
}

print(f"{'Name':<18}│ {'ID':<7}│ Token")
print("────────────────────┼─────────┼──────────────────────")
for name, tid in relevant_ids.items():
    tok_str = tok.convert_ids_to_tokens([tid], skip_special_tokens=False)[0]
    print(f"{name:<18}│ {tid:<7}│ {repr(tok_str)}")


Name              │ ID     │ Token
────────────────────┼─────────┼──────────────────────
TRUE_ID           │ 3082   │ 'ĠTrue'
FALSE_ID          │ 3641   │ 'ĠFalse'
COT_BEGIN_ID      │ 393    │ 'ĠP'
COT_END_ID        │ 362    │ 'ĠA'
pad_token_id      │ 128009 │ '<|eot_id|>'
bos_token_id      │ 128000 │ '<|begin_of_text|>'
space_id (220)    │ 220    │ 'Ġ'


In [ ]:
# AI GENERATED DEBUG CELL
from itertools import islice

N_EXAMPLES = 2          # how many rows of the HF dataset to inspect
MAX_COLS   = 120        # crop very long prompts for readability

for ex_idx in range(N_EXAMPLES):
    ids   = train_ds[ex_idx]['input_ids']
    lbls  = train_ds[ex_idx]['labels']
    toks  = tok.convert_ids_to_tokens(ids, skip_special_tokens=False)

    print(f"\n── Example {ex_idx} ──")
    print("pos | token                    | id         | label_id")
    print("----+---------------------------+------------+----------")
    for pos, (tok_str, tok_id, lab_id) in enumerate(zip(toks, ids, lbls)):
        print(f"{pos:3d} | {tok_str:<25.25} | {tok_id:<10d} | {lab_id}")
        if lab_id != -100 and lab_id != tok_id:
            raise ValueError(
                f"Label/input mismatch at pos {pos}: "
                f"input={tok_id}, label={lab_id}"
            )

print("\n✓ All inspected examples have self-consistent masks.")


In [15]:
# AI GENERATED CELL
import torch
from itertools import islice

N_TO_SHOW = 1           # how many examples to dump

def show_eval_io(model, ds, n=N_TO_SHOW):
    """
    Prints, for n examples:
      • input token, input id
      • predicted token, predicted id
      • expected (last prompt token) token & id
    """
    for ex_num, ex in enumerate(islice(ds.shuffle(seed=0), n)):
        ids  = torch.tensor(ex["input_ids"]).unsqueeze(0).to(model.device)
        mask = torch.ones_like(ids)

        # === generate one token (same args as in your accuracy()) ============
        out = model.generate(
            input_ids       = ids,
            attention_mask  = mask,
            max_new_tokens  = 1,
            do_sample       = False,
            temperature     = None,
            top_p           = None,
        )

        pred_id  = out[0, -1].item()
        exp_id   = ids[0, -1].item()

        pred_tok = tok.convert_ids_to_tokens([pred_id],  skip_special_tokens=False)[0]
        exp_tok  = tok.convert_ids_to_tokens([exp_id],   skip_special_tokens=False)[0]

        # === pretty print ====================================================
        print(f"\n━━ Example {ex_num} ━━")
        print("pos │ token                    │ id")
        print("───┼───────────────────────────┼────────")
        for pos, tok_id in enumerate(ids[0].tolist()):
            tok_str = tok.convert_ids_to_tokens([tok_id], skip_special_tokens=False)[0]
            print(f"{pos:3d} │ {tok_str:<25.25} │ {tok_id}")

        print("───┴───────────────────────────┴────────")
        print(f"Expected (last prompt token): {exp_tok!r:25} id={exp_id}")
        print(f"Predicted by model          : {pred_tok!r:25} id={pred_id}")
        verdict = "✓ MATCH" if pred_id == exp_id else "✗ MISMATCH"
        print("Result:", verdict)

# call it
show_eval_io(model, test_ds, n=N_TO_SHOW)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



━━ Example 0 ━━
pos │ token                    │ id
───┼───────────────────────────┼────────
  0 │ <|begin_of_text|>         │ 128000
  1 │ Ġ                         │ 220
  2 │ 387                       │ 20062
  3 │ Ġ                         │ 220
  4 │ 251                       │ 13860
  5 │ Ġ                         │ 220
  6 │ 946                       │ 26491
  7 │ Ġ                         │ 220
  8 │ 950                       │ 15862
  9 │ Ġ                         │ 220
 10 │ 803                       │ 20899
 11 │ Ġ                         │ 220
 12 │ 710                       │ 19027
 13 │ Ġ                         │ 220
 14 │ 600                       │ 5067
 15 │ Ġ                         │ 220
 16 │ 268                       │ 16332
 17 │ Ġ                         │ 220
 18 │ 347                       │ 17678
 19 │ Ġ                         │ 220
 20 │ 810                       │ 19232
 21 │ Ġ                         │ 220
 22 │ 008                       │ 11436
 23 │ Ġ 